# Data setup and EDA

Runs the manuscript's data steps on Colab: download the three datasets, check the folder labels,
build the 70/15/15 splits and run the EDA (section 3.5).

Before starting: **Runtime > Change runtime type > GPU** (needed later for training, not for this notebook),
and have your Kaggle API token (`kaggle.json`, from Kaggle > Settings > Create New Token) ready.

## 1. Get the code
If the repo is private, use a GitHub personal access token in the URL: `https://<token>@github.com/...`

In [ ]:
BRANCH = "main"
!git clone -b $BRANCH https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd repo
!pip install -q -e ".[train,dev]" kaggle

## 2. Optional: keep the datasets on Google Drive
The datasets are several GB. Linking `data/raw` to Drive means you only download them once.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/thesis-data/raw
!rm -rf data/raw && ln -s /content/drive/MyDrive/thesis-data/raw data/raw

## 3. Kaggle token

In [ ]:
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 4. Download

In [ ]:
!python scripts/download_datasets.py

## 5. Check the folder labels
Every folder should show `human` or `ai`. If any show `UNMATCHED` or the wrong label,
fix the `classes` rules for that dataset in `configs/default.yaml` and re-run this cell.

In [ ]:
!python scripts/inspect_datasets.py

## 6. Build the splits

In [ ]:
!python scripts/prepare_data.py

## 7. EDA
Writes `outputs/eda/report.json` (class balance, image sizes, intensity/contrast, normalization
mean/std, leakage check) and `outputs/eda/per_image.csv`. Copy `train_normalization` into
`preprocess.mean` / `preprocess.std` in the config.

In [ ]:
!python scripts/eda.py --max-per-split 2000

In [ ]:
import json, pandas as pd
report = json.load(open("outputs/eda/report.json"))
print(json.dumps(report["class_balance"], indent=1))
print("train normalization:", report["train_normalization"])
print("cross-split duplicates:", report["cross_split_duplicates"])
df = pd.read_csv("outputs/eda/per_image.csv")
df.groupby(["source", "label"])[["width", "height", "intensity", "contrast"]].describe().T

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, name in [(0, "human"), (1, "ai")]:
    d = df[df.label == label]
    axes[0].hist(d.intensity, bins=40, alpha=0.6, label=name)
    axes[1].hist(d.contrast, bins=40, alpha=0.6, label=name)
axes[0].set_title("Mean intensity"); axes[1].set_title("Contrast (luminance std)")
axes[0].legend(); plt.tight_layout()